In [190]:
import pandas as pd
import numpy as np
from pathlib import Path

## Part zero configuration

In [191]:
ROOT_DIR=Path.cwd().parent.parent.parent
DATASET_DIR= ROOT_DIR / "dataset"
CSV_FILE_PATH = DATASET_DIR / "raw_loan_dataset.csv"
print(ROOT_DIR)
print(DATASET_DIR)
print(CSV_FILE_PATH)

/home/mdev/Documents/ml/ds-ml-bootcamp
/home/mdev/Documents/ml/ds-ml-bootcamp/dataset
/home/mdev/Documents/ml/ds-ml-bootcamp/dataset/raw_loan_dataset.csv


## Load & Inspect

In [192]:
df = pd.read_csv(CSV_FILE_PATH)
df = df.rename(columns={'Approved':'Status'})
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Status
0,108810,537.0,1.1,25800,Yes,No,No
1,96482,524.0,15.0,11200,Y,No,Yes
2,28478,NaN,5.4,12100,No,No,Yes
3,"$25,851",561.0,17.6,7000,Yes,No,Yes
4,38396,527.0,9.8,10700,No,No,Approved


In [193]:
rows,columns = df.shape
print("ROWS:",rows)
print("COLUMNS:",columns)

ROWS: 103
COLUMNS: 7


In [194]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Status            103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB


In [195]:
df.isna().sum()

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Status              0
dtype: int64

In [196]:
def display_unique_values(col:str):
    print(f"\n{col}:",df[col].unique())

In [197]:
categorical_cols = ['Status','HasCollateral','PreviousDefaults']
for col in categorical_cols:
    display_unique_values(col) # use to capture and understand typo errors and in consistence category


Status: ['No' 'Yes' 'Approved' 'Rejected' 'approved' 'rejected' 'YES' 'NO']

HasCollateral: ['Yes' 'Y' 'No' 'N' nan 'yse' 'yes']

PreviousDefaults: ['No' nan 'Yes' '1' '0' 'Y' 'N']


## Clean Currency Formatting

In [198]:
def clean_numeric_col(col:str):
    print(f"\nCleaning column: {col}")
    df[col] = df[col].replace(r'[^0-9.]','',regex=True).astype(str)
    df[col] = df[col].str.strip()
    df[col] = df[col].replace('',np.nan)
    df[col] = pd.to_numeric(df[col],errors="coerce")

In [199]:
currency_cols = ['Income','LoanAmount']
for col in currency_cols:
    clean_numeric_col(col)
    print(df[col].head())


Cleaning column: Income
0    108810.0
1     96482.0
2     28478.0
3     25851.0
4     38396.0
Name: Income, dtype: float64

Cleaning column: LoanAmount
0    25800.0
1    11200.0
2    12100.0
3     7000.0
4    10700.0
Name: LoanAmount, dtype: float64


## Fix Category Errors before Imputation

In [200]:
categorical_map = {
    '1':'Yes','Approved':'Yes','approved':'Yes','Y':'Yes','yse':'Yes',
    'Rejected':'No','rejected':'No','N':'No','0':'No'
    
}

def clean_format_categorical_col(col:str,categorical_map:dict):
    print(f"\nCleaning column: {col}")
    df[col] = df[col].replace(categorical_map).astype(str)
    df[col] = df[col].str.strip().str.title()
    df[col] = df[col].replace({'Nan':np.nan})

In [201]:
for col in categorical_cols:
    display_unique_values(col)
    clean_format_categorical_col(col,categorical_map)
    display_unique_values(col)


Status: ['No' 'Yes' 'Approved' 'Rejected' 'approved' 'rejected' 'YES' 'NO']

Cleaning column: Status

Status: ['No' 'Yes']

HasCollateral: ['Yes' 'Y' 'No' 'N' nan 'yse' 'yes']

Cleaning column: HasCollateral

HasCollateral: ['Yes' 'No' nan]

PreviousDefaults: ['No' nan 'Yes' '1' '0' 'Y' 'N']

Cleaning column: PreviousDefaults

PreviousDefaults: ['No' nan 'Yes']


## Impute Missing Values

In [202]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     float64
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     float64
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Status            103 non-null    object 
dtypes: float64(4), object(3)
memory usage: 5.8+ KB


In [203]:
df.isna().sum()

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Status              0
dtype: int64

In [204]:
def fill_with_median(col:str):
    print(f"\nFilling column: {col}")
    df[col] = df[col].fillna(df[col].median())
        
def fill_with_mode(col:str):
    print(f"\nFilling column: {col}")
    df[col] = df[col].fillna(df[col].mode()[0])

In [205]:
numeric_cols = currency_cols + ['CreditScore','EmploymentYears']
print(currency_cols)
print(numeric_cols)

['Income', 'LoanAmount']
['Income', 'LoanAmount', 'CreditScore', 'EmploymentYears']


In [206]:
for col in numeric_cols:
    fill_with_median(col)


Filling column: Income

Filling column: LoanAmount

Filling column: CreditScore

Filling column: EmploymentYears


In [207]:
for col in categorical_cols:
    fill_with_mode(col)
    display_unique_values(col)


Filling column: Status

Status: ['No' 'Yes']

Filling column: HasCollateral

HasCollateral: ['Yes' 'No']

Filling column: PreviousDefaults

PreviousDefaults: ['No' 'Yes']


In [208]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            103 non-null    float64
 1   CreditScore       103 non-null    float64
 2   EmploymentYears   103 non-null    float64
 3   LoanAmount        103 non-null    float64
 4   HasCollateral     103 non-null    object 
 5   PreviousDefaults  103 non-null    object 
 6   Status            103 non-null    object 
dtypes: float64(4), object(3)
memory usage: 5.8+ KB


In [209]:
df.isna().sum()

Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Status              0
dtype: int64

## Remove Duplicates

In [210]:
before = df.shape
df = df.drop_duplicates()
print(f"Before dropping {before}(rows,cols) after dropped {df.shape}(rows,cols)")

Before dropping (103, 7)(rows,cols) after dropped (100, 7)(rows,cols)


## Outliers (IQR capping)

In [211]:
def iqr_fun(col:pd.Series,k=1.5):
    q1,q3 = col.quantile([0.25,0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

In [212]:
for col in numeric_cols:
    print(f"\nCapping column: {col}")
    lower,upper = iqr_fun(df[col])
    print("lower and upper before capping:(lower,upper)",(df[col].min(),df[col].max()))
    print("lower and upper of iqr:(lower,upper)",(lower,upper))
    df[col] = df[col].clip(lower=lower,upper=upper)
    print("lower and upper after capping:(lower,upper)",(df[col].min(),df[col].max()))


Capping column: Income
lower and upper before capping:(lower,upper) (25851.0, 250000.0)
lower and upper of iqr:(lower,upper) (-23827.875, 167619.125)
lower and upper after capping:(lower,upper) (25851.0, 167619.125)

Capping column: LoanAmount
lower and upper before capping:(lower,upper) (5000.0, 180000.0)
lower and upper of iqr:(lower,upper) (-16687.5, 66212.5)
lower and upper after capping:(lower,upper) (5000.0, 66212.5)

Capping column: CreditScore
lower and upper before capping:(lower,upper) (484.0, 920.0)
lower and upper of iqr:(lower,upper) (344.25, 962.25)
lower and upper after capping:(lower,upper) (484.0, 920.0)

Capping column: EmploymentYears
lower and upper before capping:(lower,upper) (0.5, 25.0)
lower and upper of iqr:(lower,upper) (-11.275, 35.525000000000006)
lower and upper after capping:(lower,upper) (0.5, 25.0)


## Label Encoding

In [213]:
for col in categorical_cols:
    print(f"\nEncoding column: {col}")
    df[col] = df[col].map({"Yes":1,"No":0}).astype(int)


Encoding column: Status

Encoding column: HasCollateral

Encoding column: PreviousDefaults


In [214]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            100 non-null    float64
 1   CreditScore       100 non-null    float64
 2   EmploymentYears   100 non-null    float64
 3   LoanAmount        100 non-null    float64
 4   HasCollateral     100 non-null    int64  
 5   PreviousDefaults  100 non-null    int64  
 6   Status            100 non-null    int64  
dtypes: float64(4), int64(3)
memory usage: 6.2 KB


In [215]:
df[categorical_cols].head()

,Status,HasCollateral,PreviousDefaults
0,0,1,0
1,1,1,0
2,1,0,0
3,1,1,0
4,1,0,0


## Class Balance Check

In [216]:
class_ratio = df['Status'].value_counts(normalize=True)
class_ratio

Status
1    0.66
0    0.34
Name: proportion, dtype: float64

In [219]:
if class_ratio.max() > 0.7:
    print("\nWarning: severely imbalanced classes")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")


Class balance OK for baseline Accuracy (both classes well represented).


## Feature Engineering (no leakage)

In [222]:
df['IncomePerYearEmployed'] = df['Income'] / df['EmploymentYears'].replace(0,np.nan)
df['IncomePerYearEmployed'].isna().sum()

np.int64(0)

In [223]:
df['IncomePerYearEmployed']

0     98918.181818
1      6432.133333
2      5273.703704
3      1468.806818
4      3917.959184
          ...     
95     5287.569721
96     2509.877049
97     6863.333333
98     5541.420455
99     5103.934426
Name: IncomePerYearEmployed, Length: 100, dtype: float64

In [230]:
df['DebtToIncome'] = (df['LoanAmount'] / df['Income'].replace(0,np.nan)) * 100
df['DebtToIncome'].isna().sum()

np.int64(0)

In [231]:
df['DebtToIncome']

0     23.711056
1     11.608383
2     42.488939
3     27.078256
4     27.867486
        ...    
95    57.565666
96    36.250225
97    25.066976
98    16.302843
99    33.243399
Name: DebtToIncome, Length: 100, dtype: float64